<a href="https://colab.research.google.com/github/a01663364-cyber/De_microincentivos_a_macroresultados_Actividad_6_Regresi-n_Lineal/blob/main/RegresionLinear.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regresión lineal con datos de canciones

Trabajaremos con `datos_super_inventados_mega_fake.csv`.

Flujo del ejercicio:

1. Cargar el CSV.
2. Crear las ponderaciones por color.
3. Hacer el `merge`.
4. Calcular la varianza.
5. Estimar una regresión con dos controles.
6. Quitar el control menos explicativo y comparar los modelos.

La variable dependiente será `calificacion` y la explicativa principal será `escuchas_previas`.

In [5]:
import pandas as pd

## 1. Cargar el dataset

In [6]:
df_canciones = pd.read_csv(
    "datos_super_inventados_mega_fake.csv"
)

df_canciones.head()

,id_observacion,cancion_artista,color,calificacion,escuchas_previas,edad_anios,nivel_energia
0,1,Saliva - Viktor Vaughn (MF DOOM),azul,2.93,14,25,5
1,2,Yo siempre contesto - Latin Mafia,magenta,3.40,20,18,5
2,3,Aguanta corazon Los invasores de nuevo leon,amarillo,3.82,18,24,1
3,4,fantasmas - Humbe,amarillo,3.24,18,19,5
4,5,Met Tonight - Zayn,magenta,3.78,18,20,1


## 2. Crear las ponderaciones y hacer el merge

Las ponderaciones serán: magenta = 0.50, azul = 0.30 y amarillo = 0.20.

In [7]:
df_ponderaciones = pd.DataFrame({
    "color": ["magenta", "azul", "amarillo"],
    "ponderacion": [0.50, 0.30, 0.20]
})

df_ponderaciones

,color,ponderacion
0,magenta,0.5
1,azul,0.3
2,amarillo,0.2


In [8]:
df_ponderado = df_canciones.merge(
    df_ponderaciones,
    on="color",
    how="left"
)

print(df_ponderado.to_string(index=False))

 id_observacion                                                       cancion_artista    color  calificacion  escuchas_previas  edad_anios  nivel_energia  ponderacion
              1                                      Saliva - Viktor Vaughn (MF DOOM)     azul          2.93                14          25              5          0.3
              2                                     Yo siempre contesto - Latin Mafia  magenta          3.40                20          18              5          0.5
              3                           Aguanta corazon Los invasores de nuevo leon amarillo          3.82                18          24              1          0.2
              4                                                     fantasmas - Humbe amarillo          3.24                18          19              5          0.2
              5                                                    Met Tonight - Zayn  magenta          3.78                18          20              1          0.

## 3. Importar las librerías estadísticas

In [16]:
%pip install -q numpy scipy statsmodels

import numpy as np
from scipy import stats
import statsmodels.formula.api as smf

## 4. Ejercicio rápido de varianza

In [10]:
varianza_general = df_ponderado["calificacion"].var(ddof=1)

print("Varianza general:", varianza_general)

Varianza general: 0.6458214598598597


In [11]:
varianza_por_color = (
    df_ponderado
    .groupby("color")["calificacion"]
    .var(ddof=1)
    .reset_index(name="varianza_calificacion")
)

varianza_por_color

,color,varianza_calificacion
0,amarillo,0.615169
1,azul,0.634124
2,magenta,0.687365


**Pregunta breve:** ¿qué color tiene mayor varianza y qué significa eso respecto a la dispersión de sus calificaciones?

## 5. Regresión lineal con dos controles

En `smf.ols()` colocamos la fórmula y el DataFrame:

`calificacion ~ escuchas_previas + edad_anios + nivel_energia`

La columna `ponderacion` no entra en la regresión. En este ejercicio se utiliza para la regla de ponderación por color, no como variable explicativa.

In [12]:
modelo_completo = smf.ols(
    formula="calificacion ~ escuchas_previas + edad_anios + nivel_energia",
    data=df_ponderado
).fit()

modelo_completo.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           calificacion   R-squared:                       0.584
Model:                            OLS   Adj. R-squared:                  0.583
Method:                 Least Squares   F-statistic:                     466.9
Date:                Thu, 24 Sep 2026   Prob (F-statistic):          2.29e-189
Time:                        22:33:32   Log-Likelihood:                -760.75
No. Observations:                1000   AIC:                             1529.
Df Residuals:                     996   BIC:                             1549.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
====================================================================================
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            2.1260      0.160     13.294      0.000       1.812       2.440
escuchas_previas     0.1041      0.003     37.427      0.000       0.099       0.110
edad_anios          -0.0045      0.007     -0.629      0.529      -0.019       0.010
nivel_energia       -0.0158      0.011     -1.373      0.170      -0.038       0.007
==============================================================================
Omnibus:                      752.797   Durbin-Watson:                   1.933
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               60.306
Skew:                          -0.007   Prob(JB):                     8.03e-14
Kurtosis:                       1.797   Cond. No.                         234.
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

## Cómo leer la tabla de regresión

| Elemento | Qué significa | Cómo se lee en esta base |
|---|---|---|
| Coeficiente beta | Cambio esperado en la calificación cuando la variable aumenta una unidad, manteniendo constantes las demás. | `escuchas_previas = 0.1041`: una escucha adicional se asocia con 0.1041 puntos más de calificación. |
| Intercepto | Calificación estimada cuando todas las variables explicativas valen cero. | `2.1260`: es el valor base estimado por el modelo. |
| Error estándar | Incertidumbre de la estimación del beta. Un valor menor indica mayor precisión. | El error estándar de `escuchas_previas` es `0.0028`. |
| t | Coeficiente dividido entre su error estándar. Mide qué tan lejos está el beta de cero. | `escuchas_previas` tiene `t = 37.4269`, muy alejado de cero. |
| p-valor | Evalúa la hipótesis de que el beta sea igual a cero. | `escuchas_previas` tiene `p < 0.001`; edad tiene `p = 0.5294`. |
| IC inferior | Límite inferior del intervalo de confianza del 95 %. | Para `escuchas_previas`, el límite inferior es `0.0986`. |
| IC superior | Límite superior del intervalo de confianza del 95 %. | Para `escuchas_previas`, el límite superior es `0.1095`. |
| R² | Proporción de la variación de la calificación que explica el modelo. | `R² = 0.5844`: el modelo explica aproximadamente 58.44 % de la variación. |
| R² ajustado | R² que considera el número de variables incluidas. | Sirve para comparar el modelo completo con el modelo sin `edad_anios`. |
| Error estándar residual | Tamaño típico del error de predicción del modelo, en puntos de calificación. | Se obtiene con `sqrt(modelo_completo.mse_resid)`; cuanto menor, mejor ajuste. |

Como regla práctica, si el intervalo de confianza incluye el cero o el p-valor es mayor o igual a 0.05, no hay evidencia estadística suficiente para afirmar que el beta sea distinto de cero. Estos resultados muestran asociaciones en una base simulada; no demuestran causalidad.

### Interpretar los coeficientes

- `escuchas_previas`: cambio en la calificación por una escucha adicional, manteniendo constantes edad y energía.
- `edad_anios`: cambio en la calificación por un año adicional, manteniendo constantes escuchas y energía.
- `nivel_energia`: cambio en la calificación por un punto adicional de energía, manteniendo constantes escuchas y edad.

¿Cuál de los dos controles parece menos explicativo? Revisaremos sus valores p y sus estadísticos t.

In [23]:
tabla_controles = pd.DataFrame({
    "coeficiente": modelo_completo.params[["edad_anios", "nivel_energia"]],
    "error_estandar": modelo_completo.bse[["edad_anios", "nivel_energia"]],
    "t": modelo_completo.tvalues[["edad_anios", "nivel_energia"]],
    "p_valor": modelo_completo.pvalues[["edad_anios", "nivel_energia"]]
}).sort_values("p_valor", ascending=False)

tabla_controles.round(4)

,coeficiente,error_estandar,t,p_valor
edad_anios,-0.0045,0.0072,-0.6292,0.5294
nivel_energia,-0.0158,0.0115,-1.3726,0.1702


## 6. Quitar el control menos explicativo

En esta base, `edad_anios` es el control menos explicativo porque tiene el valor p más alto y el estadístico t más cercano a cero.

In [39]:
modelo_reducido = smf.ols(
    formula="calificacion ~ escuchas_previas + nivel_energia",
    data=df_ponderado
).fit()

modelo_reducido.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:           calificacion   R-squared:                       0.584
Model:                            OLS   Adj. R-squared:                  0.583
Method:                 Least Squares   F-statistic:                     700.6
Date:                Thu, 24 Sep 2026   Prob (F-statistic):          9.35e-191
Time:                        22:52:25   Log-Likelihood:                -760.95
No. Observations:                1000   AIC:                             1528.
Df Residuals:                     997   BIC:                             1543.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
====================================================================================
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            2.0297      0.046     44.163      0.000       1.939       2.120
escuchas_previas     0.1041      0.003     37.433      0.000       0.099       0.110
nivel_energia       -0.0159      0.011     -1.386      0.166      -0.038       0.007
==============================================================================
Omnibus:                      741.622   Durbin-Watson:                   1.931
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               60.123
Skew:                          -0.008   Prob(JB):                     8.80e-14
Kurtosis:                       1.799   Cond. No.                         33.7
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [41]:
comparacion = pd.DataFrame({
    "Modelo": ["Completo", "Sin edad_anios"],
    "R2": [
        modelo_completo.rsquared,
        modelo_reducido.rsquared
    ],
    "R2 ajustado": [
        modelo_completo.rsquared_adj,
        modelo_reducido.rsquared_adj
    ],
    "Error estándar residual": [
        np.sqrt(modelo_completo.mse_resid),
        np.sqrt(modelo_reducido.mse_resid)
    ],
    "Beta escuchas_previas": [
        modelo_completo.params["escuchas_previas"],
        modelo_reducido.params["escuchas_previas"]
    ]
})

comparacion.round(4)

,Modelo,R2,R2 ajustado,Error estándar residual,Beta escuchas_previas
0,Completo,0.5844,0.5832,0.5188,0.1041
1,Sin edad_anios,0.5843,0.5834,0.5187,0.1041


### Interpretación final

1. ¿Cambió mucho el coeficiente de `escuchas_previas` al quitar `edad_anios`?
    
        R= No, el cambio al quitar edad_anios fue extremadamente mínimo.

2. ¿Qué ocurrió con el R²?

        R= Al quitar edad_anios disminuyo una diezmilésima.

3. ¿Qué ocurrió con el R² ajustado?

        R= Al quitar edad_anios aumento 2 diezmilésimas.

4. ¿Qué modelo elegirías y por qué?

        R= Elejiría el modelo sin edad_anios solo por el motivo de que este es 2 diezmilésimas mas exacto.
